# CineEmbed — 01 Smoke Test

Validates that the `cineembed` package's backbone, three heads, losses, and eval
helpers all run end-to-end on a 200-row synthetic batch. Should complete in <1 minute
on Mac CPU.

In [ ]:
import sys, os
from pathlib import Path
# Local Mac path; for Colab replace with /content/cineembed-repo
REPO_ROOT = Path('..').resolve() if Path('../pyproject.toml').exists() else Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch

from cineembed import data, backbone, heads, losses, eval as cev, train

print(f"cineembed loaded from {REPO_ROOT / 'src'}")
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")

In [ ]:
BLOCK_DIMS = {'numerical': 6, 'genre': 22, 'language': 31, 'decade': 2,
              'awards': 6, 'text': 384, 'director': 113}
PROJ_DIMS = {'numerical': 16, 'genre': 16, 'language': 16, 'decade': 4,
             'awards': 16, 'text': 64, 'director': 32}

# Reuse the conftest synthetic-matrix logic inline
rng = np.random.default_rng(42)
n = 200
total = sum(BLOCK_DIMS.values())
X = np.zeros((n, total), dtype=np.float32)
slices, start = {}, 0
for b, d in BLOCK_DIMS.items():
    slices[b] = slice(start, start + d)
    start += d

# Fill blocks (simplified — production uses richer synthetic data)
X[:, slices['numerical']] = rng.standard_normal((n, 6)).astype(np.float32)
for i in range(n):
    cols = rng.choice(BLOCK_DIMS['genre'], size=3, replace=False)
    X[i, slices['genre'].start + cols] = 1.0
text_raw = rng.standard_normal((n, 384)).astype(np.float32)
X[:, slices['text']] = text_raw / np.linalg.norm(text_raw, axis=1, keepdims=True).clip(1e-8)
has_bio = (rng.uniform(0, 1, size=n) < 0.05).astype(np.float32)
X[:, slices['director'].start + 64] = has_bio  # has_director_bio col

X_t = torch.from_numpy(X)
has_bio_t = torch.from_numpy(has_bio)
print(f"X shape: {X.shape}; has_bio sum: {int(has_bio.sum())}")

In [ ]:
torch.manual_seed(42)
bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=64)
ae = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
vae = heads.VAEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
dec_head = heads.DECHead(bb, ae.decoder, n_clusters=10, latent_dim=64)

print(f"AE   params: {sum(p.numel() for p in ae.parameters()):>7,}")
print(f"VAE  params: {sum(p.numel() for p in vae.parameters()):>7,}")
print(f"DEC  params: {sum(p.numel() for p in dec_head.parameters()):>7,}")

In [ ]:
blocks = {b: X_t[:, slc] for b, slc in slices.items()}

decoded = ae(blocks)
print(f"AE  decoded keys: {sorted(decoded.keys())}")
print(f"    decoded['numerical'].shape: {decoded['numerical'].shape}")

decoded_v, mu, log_var = vae(blocks)
print(f"VAE mu, log_var shapes: {mu.shape}, {log_var.shape}")

# Initialize DEC centers from AE encoder output
with torch.no_grad():
    z_init = bb(blocks).numpy()
dec_head.initialize_centers(z_init, seed=42)
z_dec, decoded_d, q = dec_head(blocks)
print(f"DEC z, q shapes: {z_dec.shape}, {q.shape}")
print(f"    q row sums (should be \u22481): min={q.sum(1).min():.4f}, max={q.sum(1).max():.4f}")

In [ ]:
w_blocks = losses.compute_block_weights(X, slices, w_min=0.1, w_max=10.0)
print(f"Block weights: {w_blocks}")

# AE / W2
loss_ae = losses.weighted_recon_loss(decoded, blocks, has_bio_t, w_blocks)
print(f"AE  W2 loss: {loss_ae.item():.4f}")

# AE / W1 baseline
loss_w1 = losses.weighted_recon_loss_uniform(decoded, blocks, has_bio_t)
print(f"AE  W1 loss: {loss_w1.item():.4f}")

# VAE ELBO with beta=0.5
loss_vae, recon_v, kl_v = losses.vae_elbo(decoded_v, blocks, mu, log_var, has_bio_t, w_blocks, beta=0.5)
print(f"VAE elbo: {loss_vae.item():.4f}  (recon={recon_v:.4f}  kl={kl_v:.4f})")

# DEC loss
loss_dec, kl_d, recon_d = losses.dec_loss(z_dec, decoded_d, blocks, dec_head.cluster_centers,
                                            has_bio_t, w_blocks, lambda_recon=0.1)
print(f"DEC loss: {loss_dec.item():.4f}  (kl={kl_d:.4f}  recon={recon_d:.4f})")

In [ ]:
synthetic_labels = {
    'primary_genre': rng.integers(0, 21, size=n),
    'decade_bin':    rng.integers(0, 13, size=n),
    'lang_top10':    rng.integers(0, 11, size=n),
}

# KMeans on AE latent
with torch.no_grad():
    z_ae = ae.encode(blocks).numpy()
c_ids = cev.cluster_assignments_kmeans(z_ae, k=10, seed=42)
metrics = cev.evaluate_run(c_ids, synthetic_labels)
print("Synthetic NMI/ARI (random labels \u2192 near-zero):")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

✅ Smoke test passed if all cells ran without error and printed non-NaN losses + per-axis NMI/ARI values.

For full data training: run `02_train_ae.ipynb`, `03_train_vae.ipynb`, `04_train_dec.ipynb` on Colab.